In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/mental-cognitive-workload-eeg-data-stew-dataset/EEG data summary.pdf
/kaggle/input/mental-cognitive-workload-eeg-data-stew-dataset/class_012.mat
/kaggle/input/mental-cognitive-workload-eeg-data-stew-dataset/dataset.mat
/kaggle/input/mental-cognitive-workload-eeg-data-stew-dataset/rating.mat
/kaggle/input/mental-cognitive-workload-eeg-data-stew-dataset/three_class_one_hot.mat


In [2]:
 import scipy.io
dataset = scipy.io.loadmat('../input/mental-cognitive-workload-eeg-data-stew-dataset/dataset.mat')
rating = scipy.io.loadmat('../input/mental-cognitive-workload-eeg-data-stew-dataset/rating.mat')
class012 = scipy.io.loadmat('../input/mental-cognitive-workload-eeg-data-stew-dataset/class_012.mat')
class012_onehot = scipy.io.loadmat('../input/mental-cognitive-workload-eeg-data-stew-dataset/three_class_one_hot.mat')

dataset = dataset['dataset'] 
rating = rating['rating']
class012 = class012['class_012']
class012_onehot = class012_onehot['three_class_one_hot']

print(dataset.shape)
print(rating.shape)
print(class012.shape)

(14, 19200, 45)
(45, 1)
(45, 1)


In [3]:
dataset.resize(45,19200,14)
print(dataset.shape)
print(class012.shape)

(45, 19200, 14)
(45, 1)


In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(dataset, class012, test_size = 0.10,random_state = 42, stratify=class012 )
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

(40, 19200, 14)
(5, 19200, 14)
(40, 1)
(5, 1)


In [5]:
print(Y_test)

[[2]
 [1]
 [1]
 [0]
 [2]]


In [6]:
import tensorflow as tf

model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Conv1D(filters=128, kernel_size=4, activation='relu', input_shape=(19200,14)))
model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Conv1D(filters=64, kernel_size=8, activation='relu'))
model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Conv1D(filters=32, kernel_size=16, activation='relu'))
model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(100, activation='relu'))
model.add(tf.keras.layers.Dense(1, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='Adam', metrics=['Accuracy'])

/opt/conda/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:

model.fit(X_train,Y_train,batch_size=1,epochs=2)

Epoch 1/2


/opt/conda/lib/python3.10/site-packages/keras/src/ops/nn.py:545: UserWarning: You are using a softmax over axis -1 of a tensor of shape (1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/keras/src/losses/losses.py:22: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(1, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)


40/40 ━━━━━━━━━━━━━━━━━━━━ 11s 185ms/step - Accuracy: 0.3691 - loss: 0.0000e+00
Epoch 2/2
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 186ms/step - Accuracy: 0.4306 - loss: 0.0000e+00


In [8]:
loss,accu = model.evaluate(X_test,Y_test)
print(loss)
print(accu)

/opt/conda/lib/python3.10/site-packages/keras/src/ops/nn.py:545: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/keras/src/losses/losses.py:22: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 421ms/step - Accuracy: 0.4000 - loss: 0.0000e+00
0.0
0.4000000059604645


In [9]:
from sklearn.metrics import confusion_matrix
predictions = model.predict(X_test)
Y_Predicted=np.argmax(predictions,axis=1)
print("Predicted Labels", Y_Predicted)
print("Actual Labels", np.transpose(Y_test))
cm = confusion_matrix(Y_test,Y_Predicted)
print("Confusion Matrix")
print(cm)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step
Predicted Labels [0 0 0 0 0]
Actual Labels [[2 1 1 0 2]]
Confusion Matrix
[[1 0 0]
 [2 0 0]
 [2 0 0]]


/opt/conda/lib/python3.10/site-packages/keras/src/ops/nn.py:545: UserWarning: You are using a softmax over axis -1 of a tensor of shape (5, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


In [10]:

cm = confusion_matrix(Y_test,Y_Predicted)
cm

array([[1, 0, 0],
       [2, 0, 0],
       [2, 0, 0]])

In [12]:
def multi_class_parameters (conf_matrix):
    # (1) accuracy
    ACC = (sum(conf_matrix.diagonal()) / sum(sum(conf_matrix)))
    # (2) mis-classification
    MC = 1- ACC
    [row, col]=conf_matrix.shape
    if (row>2):
        PR=[None] * row
        RC=[None] * row
        F1=[None] * row
        SP=[None] * row
        ind= list(range(0, row))
        for cm in range(0, row):
            TN=None
            ind= list(range(0, row))
            PR[cm]=conf_matrix[cm,cm]/(conf_matrix[:,cm]).sum()
            RC[cm]=conf_matrix[cm,cm]/(conf_matrix[cm,:]).sum()
            F1[cm]=(2*(PR[cm])*(RC[cm]))/((PR[cm])+(RC[cm]))
            ind.pop(cm)
            TN=(conf_matrix[np.ix_(ind,ind)]).sum()
            SP[cm]=TN/(conf_matrix[ind,:]).sum()
        # (3) Sensitivity (Recall)
        Avg_RC = sum(RC)/row
        # (4) Specificity
        Avg_SP = sum(SP)/row
        # (5) Precision
        Avg_PR = sum(PR)/row
        # (6) f_1 score
        Avg_F1 = sum(F1)/row
        print('1. Accuracy:', ACC)
        print('2. Mis-Classification:', MC)
        print('3. Sensitivity (Recall):', Avg_RC)
        print('4. Specificity:', Avg_SP)
        print('5. Precision:', Avg_PR)
        print('6. f_1 Score:', Avg_F1)
    else:
        TP = conf_matrix[1][1]
        TN = conf_matrix[0][0]
        FP = conf_matrix[0][1]
        FN = conf_matrix[1][0]
        MC = 1- ACC
        # (1) accuracy
        ACC = (float (TP+TN) / float(TP + TN + FP + FN))
        # (2) mis-classification
        # (3) sensitivity (Recall)
        RC = (TP / float(TP + FN))
        # (4) specificity
        SP = (TN / float(TN + FP))
        # (5) precision
        PR = (TP / float(TP + FP))
        # (6) f_1 score
        F1 = 2 * ((PR * RC) / (PR + RC))
        print('1. Accuracy:', ACC)
        print('2. Mis-Classification:', MC)
        print('3. Sensitivity (Recall):', RC)
        print('4. Specificity:', SP)
        print('5. Precision:', PR)
        print('6. f_1 Score:', F1)



In [13]:
multi_class_parameters(cm)

1. Accuracy: 0.2
2. Mis-Classification: 0.8
3. Sensitivity (Recall): 0.3333333333333333
4. Specificity: 0.6666666666666666
5. Precision: nan
6. f_1 Score: nan


/tmp/ipykernel_33/2178189655.py:16: RuntimeWarning: invalid value encountered in scalar divide
  PR[cm]=conf_matrix[cm,cm]/(conf_matrix[:,cm]).sum()
